# Aula 19: Algoritmo de Kruskal e Prim - Aplicações no Sistema Visão-AGV

## 1. Visão Geral da Aula

Nesta aula, exploraremos a implementação e aplicação dos algoritmos de Kruskal e Prim para a resolução do problema da **Árvore Geradora Mínima (Minimum Spanning Tree - MST)** dentro do contexto da intralogística e do projeto Visão-AGV.

O objetivo central da MST é conectar um conjunto de nós (pontos de interesse, zonas de operação, ou estações de recarga) garantindo conectividade total (todos os nós interligados) sem a formação de ciclos fechados, e minimizando o custo total da infraestrutura (ex: metros de cabo magnético, fibra óptica, ou custo de comunicação).

### Casos de Uso no Projeto Visão-AGV

1.  **Infraestrutura Física (Kruskal/Prim):** Se o galpão necessitar de uma malha de guias magnéticas ou cabos de energia interligando todas as zonas de operação do AGV, a MST fornece a topologia de menor custo total de instalação.
2.  **Rede de Comunicação Sem Fio (MST via Prim):** Em um ambiente industrial ruidoso, conectar pontos de acesso Wi-Fi ou balizas de comunicação para cobrir todas as células de manufatura exige minimizar a latência e o custo de instalação dos repetidores. A MST desenha o *backbone* mais econômico para a rede de telemetria do AGV.
3.  **Clustering Espacial (Variação de Kruskal):** Ao agrupar nós geograficamente próximos para definir 'zonas de atuação' de frotas de AGVs, podemos interromper o algoritmo de Kruskal prematuramente, criando sub-árvores isoladas (florestas) que representam clusters lógicos da fábrica.

## 2. Abordagem Prática

Implementaremos as duas abordagens clássicas para encontrar a MST:

-   **Algoritmo de Kruskal:** Foca nas *arestas*. Ordena os corredores do menor para o maior custo e os adiciona à árvore, desde que não formem ciclos (utilizando a estrutura de dados *Union-Find* / *Disjoint Sets*). É excelente para grafos esparsos e modelagem global da malha.
-   **Algoritmo de Prim:** Foca nos *vértices*. Começa em um nó arbitrário (ex: Estação Base) e expande a fronteira selecionando sempre o corredor mais barato que conecta a árvore a um novo nó (utilizando uma fila de prioridade Min-Heap). É muito intuitivo para cenários de expansão de rede a partir de um ponto central.

---

### Estrutura do Notebook

1.  **Definição da Malha:** Criação do grafo ponderado representando a malha logística (nós e corredores).
2.  **Implementação de Kruskal:** Classe `DisjointSet` e função principal do algoritmo.
3.  **Implementação de Prim:** Utilização de `heapq` para seleção gulosa a partir da Estação Base.
4.  **Comparativo e Visualização:** Análise dos resultados e (opcionalmente, se a biblioteca `networkx` estiver disponível) renderização visual da MST extraída do layout original.


In [1]:
# Setup Inicial e Definição da Malha de Navegação

import heapq

# Representação do Grafo (Adjacency List) para a Malha do AGV
# Formato: grafo[origem] = {destino: custo_metros}
malha_agv = {
    'ST-01_Base': {'DOC-101': 15, 'ALM-201': 20, 'ZON-A': 35},
    'DOC-101': {'ST-01_Base': 15, 'ALM-201': 10, 'ZON-B': 25},
    'ALM-201': {'ST-01_Base': 20, 'DOC-101': 10, 'ZON-A': 30, 'AMO-301': 12},
    'ZON-A': {'ST-01_Base': 35, 'ALM-201': 30, 'R-101': 18},
    'ZON-B': {'DOC-101': 25, 'AMO-301': 22, 'DEP-401': 14},
    'AMO-301': {'ALM-201': 12, 'ZON-B': 22, 'R-101': 8, 'DEP-401': 16},
    'R-101': {'ZON-A': 18, 'AMO-301': 8, 'DEP-401': 40},
    'DEP-401': {'ZON-B': 14, 'AMO-301': 16, 'R-101': 40}
}

# Função auxiliar para extrair lista de arestas únicas (necessário para Kruskal)
def extrair_arestas_unicas(grafo):
    arestas = []
    visitados = set()
    for origem, conexoes in grafo.items():
        for destino, peso in conexoes.items():
            # Cria uma chave única para a aresta (não direcionada) para evitar duplicatas
            aresta_id = tuple(sorted([origem, destino]))
            if aresta_id not in visitados:
                arestas.append((peso, origem, destino))
                visitados.add(aresta_id)
    return arestas

print("Malha do AGV inicializada. Total de corredores:", len(extrair_arestas_unicas(malha_agv)))


## 3. Implementação do Algoritmo de Kruskal

O Algoritmo de Kruskal exige uma estrutura para gerenciar conjuntos disjuntos (Union-Find) e detectar ciclos eficientemente.

In [2]:
# Estrutura de Dados Union-Find (Disjoint Set) com Path Compression e Union by Rank
class UnionFind:
    def __init__(self, vertices):
        self.parent = {v: v for v in vertices}
        self.rank = {v: 0 for v in vertices}

    def find(self, item):
        if self.parent[item] == item:
            return item
        # Path compression: achata a árvore ligando diretamente à raiz
        self.parent[item] = self.find(self.parent[item])
        return self.parent[item]

    def union(self, root1, root2):
        # Union by rank: anexa a árvore menor na raiz da maior
        if self.rank[root1] > self.rank[root2]:
            self.parent[root2] = root1
        elif self.rank[root1] < self.rank[root2]:
            self.parent[root1] = root2
        else:
            self.parent[root2] = root1
            self.rank[root1] += 1

def algoritmo_kruskal(grafo):
    """
    Calcula a Árvore Geradora Mínima usando Kruskal.
    Ideal para planejar o cabeamento magnético de menor custo global.
    """
    mst = []
    custo_total = 0
    
    # 1. Obter todas as arestas e ordená-las por peso (ordem crescente)
    arestas = extrair_arestas_unicas(grafo)
    arestas.sort(key=lambda item: item[0])
    
    # 2. Inicializar a estrutura Union-Find com todos os vértices
    uf = UnionFind(grafo.keys())
    
    # 3. Iterar sobre as arestas ordenadas
    for peso, origem, destino in arestas:
        raiz_origem = uf.find(origem)
        raiz_destino = uf.find(destino)
        
        # Se as raízes são diferentes, a aresta não forma ciclo
        if raiz_origem != raiz_destino:
            uf.union(raiz_origem, raiz_destino)
            mst.append((origem, destino, peso))
            custo_total += peso
            
            # Otimização: A MST de um grafo conexo com V vértices tem V-1 arestas
            if len(mst) == len(grafo) - 1:
                break
                
    return mst, custo_total

# Execução de Kruskal na Malha
mst_kruskal, custo_kruskal = algoritmo_kruskal(malha_agv)

print("--- RESULTADO KRUSKAL ---")
print(f"Custo total da infraestrutura: {custo_kruskal} metros")
print("Corredores Selecionados (Origem, Destino, Metros):")
for corredor in mst_kruskal:
    print(f"  {corredor[0]} <---> {corredor[1]} (Custo: {corredor[2]})")


## 4. Implementação do Algoritmo de Prim

O Algoritmo de Prim cresce a árvore a partir de um nó central (ex: Estação de Recarga Base). Utilizaremos uma fila de prioridade (`min-heap`) para extrair sempre a aresta de menor custo que conecta a malha existente a um novo nó.

In [3]:
def algoritmo_prim(grafo, no_inicial):
    """
    Calcula a Árvore Geradora Mínima usando Prim.
    Simula a expansão da rede de comunicação a partir de um ponto central.
    """
    mst = []
    custo_total = 0
    
    # Conjunto para rastrear nós já inclusos na árvore
    visitados = set([no_inicial])
    
    # Min-heap (fila de prioridade) armazenando (peso, origem, destino)
    arestas_borda = []
    
    # Inicializa a fronteira com os vizinhos do nó inicial
    for vizinho, peso in grafo[no_inicial].items():
        heapq.heappush(arestas_borda, (peso, no_inicial, vizinho))
        
    # Enquanto a árvore não incluir todos os nós e a fronteira não estiver vazia
    while arestas_borda and len(visitados) < len(grafo):
        # Extrai a aresta mais barata da fronteira
        peso, origem, destino = heapq.heappop(arestas_borda)
        
        # Se o destino ainda não está na árvore (evita ciclos)
        if destino not in visitados:
            visitados.add(destino)
            mst.append((origem, destino, peso))
            custo_total += peso
            
            # Adiciona os novos corredores vizinhos do destino à fronteira
            for novo_vizinho, novo_peso in grafo[destino].items():
                if novo_vizinho not in visitados:
                    heapq.heappush(arestas_borda, (novo_peso, destino, novo_vizinho))
                    
    return mst, custo_total

# Execução de Prim a partir da Estação Base
no_partida = 'ST-01_Base'
mst_prim, custo_prim = algoritmo_prim(malha_agv, no_partida)

print("--- RESULTADO PRIM ---")
print(f"Expansão iniciada a partir de: {no_partida}")
print(f"Custo total da infraestrutura: {custo_prim} metros")
print("Corredores Selecionados (Expansão Passo a Passo):")
for passo, corredor in enumerate(mst_prim, 1):
    print(f"  Passo {passo}: {corredor[0]} ---> {corredor[1]} (Custo: {corredor[2]})")


## 5. Conclusão e Insights

1.  **Garantia Teórica:** Note que tanto o algoritmo de Kruskal quanto o algoritmo de Prim retornam o **mesmo custo total** da Árvore Geradora Mínima (se os pesos forem únicos, retornam também a mesma topologia exata).
2.  **Aplicação no SCADA:** Se o gestor da fábrica decidir instalar trilhos magnéticos para o AGV operar de forma mais estável nas áreas críticas, os resultados acima ditam exatamente quais trechos físicos devem ser asfaltados/marcados, garantindo que o AGV chegue a qualquer setor com o mínimo de material gasto.